# 05 — Adenosine production by Spp1 / Trem2 macrophage gates

**Question.** Of the two extracellular-adenosine routes — the canonical CD39 → CD73 route and the
alternative CD38 → CD203a/ENPP1 → CD73 route — how much of each enzymatic step is carried by
**Spp1⁺**, **Trem2⁺** and **Spp1⁺Trem2⁺** macrophages?

This re-cuts the macrophage compartment by **marker positivity** (what a PhenoCycler panel would
gate on) rather than by transcriptome state, so the answer transfers to protein.

* Gate = detection of the transcript in that cell (count ≥ 1), computed within macrophages
  (`is_mac`, 48,475 cells). Four gates, plus all non-macrophage cells as "Other cells".
* **Contribution** = the gate's share of the whole tumor's transcripts for that enzyme, a
  partition that sums to 1 and so folds in abundance as well as per-cell expression.
* **Enrichment** = log2(gene share ÷ the gate's share of all UMI). Zero means the gate holds the
  enzyme exactly in proportion to its transcriptional mass, i.e. no specialization.
* Unit of analysis = **mouse** (treated arms only, 12 mice); CIs are t-based over mice and the
  p-value is an exact sign-flip test over mice.

Outputs → `results/rt_advantage/specific_cell_type/adenosine/`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import itertools, time, json
from pathlib import Path
import numpy as np, pandas as pd, h5py, anndata as ad
import scipy.sparse as sp
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from scipy import stats

_HPC = Path("/coh_labs/yunroseli/Jona/CAR-T"); _LOC = Path("/Users/janzules/Roselab/Spatial/CAR_T")
PROJ = _LOC if _LOC.exists() else _HPC
SC_REF = (PROJ/"data/sc-reference/annotation_v4.h5ad" if PROJ is _LOC
          else PROJ/"data/zarr/fullDataset/annotating_references/annotation_v4.h5ad")
MHC = PROJ/"results/mhc_cytotox"
OUT = PROJ/"results/rt_advantage/specific_cell_type/adenosine"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 0

INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
C_DP, C_SPP1, C_TREM2, C_DN, C_OTH = "#5342ba", "#d75853", "#3987e5", "#b9b7ae", "#e1e0d9"
GATES = ["Spp1+Trem2+", "Spp1+Trem2-", "Spp1-Trem2+", "Spp1-Trem2-", "Other cells"]
GLAB = {"Spp1+Trem2+": "Spp1⁺Trem2⁺", "Spp1+Trem2-": "Spp1⁺Trem2⁻",
        "Spp1-Trem2+": "Spp1⁻Trem2⁺", "Spp1-Trem2-": "Spp1⁻Trem2⁻", "Other cells": "Other cells"}
GCOL = dict(zip(GATES, [C_DP, C_SPP1, C_TREM2, C_DN, C_OTH]))
ENZ = ["Entpd1", "Cd38", "Enpp1", "Nt5e"]
ELAB = {"Entpd1": "CD39\nENTPD1", "Cd38": "CD38", "Enpp1": "CD203a\nENPP1", "Nt5e": "CD73\nNT5E"}
ESTEP = {"Entpd1": "ATP/ADP → AMP", "Cd38": "NAD⁺ → ADPR", "Enpp1": "ADPR → AMP", "Nt5e": "AMP → adenosine"}

plt.rcParams.update({
    "font.family": ["Arial", "DejaVu Sans"],
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False, "lines.solid_capstyle": "round",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none", "savefig.dpi": 300, "text.color": INK})


def save_figure(fig, stem):
    for ext in ("pdf", "png"):
        fig.savefig(OUT/f"{stem}.{ext}", bbox_inches="tight")
    print(f"saved {OUT/stem}.pdf / .png")

## 1. Cells, gates and the per-cell enzyme counts

In [2]:
t0 = time.time()
with h5py.File(SC_REF, "r") as f:
    obs = ad.io.read_elem(f["obs"]); var = ad.io.read_elem(f["var"])
    C = sp.csr_matrix(ad.io.read_elem(f["layers/counts"]))
genes = pd.Index(var.index.astype(str))
meta = pd.read_parquet(MHC/"cell_panel.parquet",
                       columns=["sample", "arm", "site", "mouse", "RT", "cart", "ct2", "ct3", "lvl4",
                                "is_mac", "SPP1_TAM", "total_counts"])
assert (meta.index == obs.index).all(), "cell order differs from the 00 cell panel"
design = pd.read_csv(MHC/"design_samples.csv").set_index("sample")
TREATED = design.index[design.arm != "NoTx"]
col = lambda g: np.asarray(C[:, genes.get_loc(g)].todense()).ravel()
X = {g: col(g) for g in ENZ}
spp1_c, trem2_c = col("Spp1"), col("Trem2")
mac = meta.is_mac.values
sp_p, tr_p = spp1_c > 0, trem2_c > 0

gate = pd.Series("Other cells", index=meta.index)
gate[mac & sp_p & tr_p] = "Spp1+Trem2+"
gate[mac & sp_p & ~tr_p] = "Spp1+Trem2-"
gate[mac & ~sp_p & tr_p] = "Spp1-Trem2+"
gate[mac & ~sp_p & ~tr_p] = "Spp1-Trem2-"
meta["gate"] = gate.values
G = gate.values

# sensitivity gate: Spp1-high = top within-sample tertile among macrophages (not bare detection)
meta["Spp1_ln"] = np.log1p(spp1_c / meta.total_counts.values * 1e4)
tert = pd.Series("", index=meta.index)
for s, d in meta[mac].groupby("sample"):
    tert.loc[d.index] = pd.qcut(d.Spp1_ln.rank(method="first"), 3, labels=["lo", "mid", "hi"]).astype(str).values
hi = (tert == "hi").values
gate2 = pd.Series("Other cells", index=meta.index)
gate2[mac & hi & tr_p] = "Spp1+Trem2+"; gate2[mac & hi & ~tr_p] = "Spp1+Trem2-"
gate2[mac & ~hi & tr_p] = "Spp1-Trem2+"; gate2[mac & ~hi & ~tr_p] = "Spp1-Trem2-"

SIZES = pd.DataFrame({"n_cells": pd.Series(G).value_counts()}).reindex(GATES)
SIZES["pct_of_macrophages"] = 100 * SIZES.n_cells / mac.sum()
SIZES.loc["Other cells", "pct_of_macrophages"] = np.nan
SIZES["pct_of_all_cells"] = 100 * SIZES.n_cells / len(meta)
print(f"{C.shape[0]:,} cells read in {time.time()-t0:.0f}s; {mac.sum():,} macrophages")
print(SIZES.round(1).to_string())

93,548 cells read in 1s; 48,475 macrophages
             n_cells  pct_of_macrophages  pct_of_all_cells
Spp1+Trem2+    20413                42.1              21.8
Spp1+Trem2-    13061                26.9              14.0
Spp1-Trem2+     5247                10.8               5.6
Spp1-Trem2-     9754                20.1              10.4
Other cells    45073                 NaN              48.2


## 2. Contribution and enrichment, with the mouse as the unit

For each treated sample: the gate's share of that sample's transcripts for the enzyme, and its
share of that sample's total UMI. Samples are averaged within mouse (two tumors per mouse), so
n = 12. `enrich` = log2(enzyme share ÷ UMI share); the p-value is an exact sign-flip test on the
12 mouse values (2¹² = 4,096 relabelings).

In [3]:
def signflip_p(v):
    v = np.asarray(v, float); v = v[~np.isnan(v)]; n = len(v)
    if n < 4: return np.nan
    S = np.array(list(itertools.product([1, -1], repeat=n))) if n <= 16 else \
        np.random.default_rng(SEED).choice([1, -1], size=(50000, n))
    null = (S * v).mean(1)
    return float((np.abs(null) >= abs(v.mean()) - 1e-12).mean())


def ci95(v):
    v = np.asarray(v, float); v = v[~np.isnan(v)]; n = len(v)
    if n < 2: return (np.nan, np.nan)
    h = stats.t.ppf(0.975, n - 1) * v.std(ddof=1) / np.sqrt(n)
    return (v.mean() - h, v.mean() + h)


keep = meta.arm.ne("NoTx").values
df = pd.DataFrame({"sample": meta["sample"].values, "mouse": meta.mouse.values, "gate": G,
                   "umi": meta.total_counts.values, **{g: X[g] for g in ENZ}})[keep]
m_of = design.mouse

share_rows, umi_rows = [], []
per_umi = df.groupby(["sample", "gate"]).umi.sum().unstack(fill_value=0).reindex(columns=GATES, fill_value=0)
UMI_SH = (per_umi.T / per_umi.sum(1)).T * 100
for g in ENZ:
    per = df.groupby(["sample", "gate"])[g].sum().unstack(fill_value=0).reindex(columns=GATES, fill_value=0)
    sh = (per.T / per.sum(1)).T * 100
    sh = sh.assign(gene=g); share_rows.append(sh)
SH = pd.concat(share_rows)
SH_M = SH.groupby([SH.gene, m_of.reindex(SH.index).values]).mean()        # mouse means
UMI_M = UMI_SH.groupby(m_of.reindex(UMI_SH.index).values).mean()

rows = []
for g in ENZ:
    sm = SH_M.loc[g]
    for k in GATES:
        e = np.log2(sm[k].values / UMI_M[k].reindex(sm.index).values)
        lo, hi_ = ci95(sm[k]); elo, ehi = ci95(e)
        rows.append(dict(gene=g, gate=k, n_mice=len(sm), share_mean=sm[k].mean(), share_sd=sm[k].std(),
                         share_lo=lo, share_hi=hi_, umi_share=UMI_M[k].mean(),
                         enrich=np.nanmean(e), enrich_lo=elo, enrich_hi=ehi, p_enrich=signflip_p(e)))
CONTRIB = pd.DataFrame(rows)
CONTRIB.to_csv(OUT/"05_gate_contribution.csv", index=False)
print(CONTRIB.pivot(index="gate", columns="gene", values="share_mean").reindex(GATES)[ENZ].round(1).to_string())
print()
print(CONTRIB.pivot(index="gate", columns="gene", values="enrich").reindex(GATES)[ENZ].round(2).to_string())

# per-cell detection, and pathway completeness, by gate
det = pd.DataFrame({"gate": G, **{g: X[g] > 0 for g in ENZ}})
det["both_upstream"] = det.Cd38 & det.Enpp1
det["alt_complete"] = det.Cd38 & det.Enpp1 & det.Nt5e
det["canon_complete"] = det.Entpd1 & det.Nt5e
DET = (det.groupby("gate").mean() * 100).reindex(GATES)
DET["n_cells"] = det.groupby("gate").size().reindex(GATES)
DET.to_csv(OUT/"05_gate_detection.csv")
print(); print(DET.round(1).to_string())

# within the double-positive gate: which transcriptome state carries which step
dp = G == "Spp1+Trem2+"
ST = pd.DataFrame({"state": meta.ct3.values[dp], **{g: X[g][dp] > 0 for g in ENZ}})
STATE = (ST.groupby("state").mean() * 100)
STATE["n_cells"] = ST.groupby("state").size()
STATE = STATE.sort_values("Enpp1", ascending=False)
STATE.to_csv(OUT/"05_dp_state_breakdown.csv")
print(); print(STATE.round(1).to_string())

# sensitivity: Spp1-high tertile instead of bare detection
g2 = gate2.values[keep]
sens = []
for g in ENZ:
    per = pd.DataFrame({"s": df["sample"].values, "g": g2, "v": df[g].values}) \
            .groupby(["s", "g"]).v.sum().unstack(fill_value=0).reindex(columns=GATES, fill_value=0)
    sh = ((per.T / per.sum(1)).T * 100).groupby(m_of.reindex(per.index).values).mean()
    sens.append(sh.mean().rename(g))
SENS = pd.concat(sens, axis=1).reindex(GATES)
SENS.to_csv(OUT/"05_sensitivity_spp1_tertile.csv")
print(); print("sensitivity — Spp1-high = top within-sample tertile:"); print(SENS.round(1).to_string())

gene         Entpd1  Cd38  Enpp1  Nt5e
gate                                  
Spp1+Trem2+    36.8  59.2   71.0  37.0
Spp1+Trem2-    13.2  11.9    9.5  15.8
Spp1-Trem2+     6.9   7.5    5.6   6.1
Spp1-Trem2-     9.5   6.1    2.5   6.9
Other cells    33.6  15.4   11.4  34.3

gene         Entpd1  Cd38  Enpp1  Nt5e
gate                                  
Spp1+Trem2+    0.11  0.88   1.15  0.19
Spp1+Trem2-   -0.12 -0.31  -0.61  0.14
Spp1-Trem2+    0.06 -0.13  -0.35 -0.14
Spp1-Trem2-   -0.08 -0.80  -2.30 -0.48
Other cells   -0.17 -1.32  -1.82 -0.11

             Entpd1  Cd38  Enpp1  Nt5e  both_upstream  alt_complete  canon_complete  n_cells
gate                                                                                        
Spp1+Trem2+    87.6  35.4   14.3  24.9            7.8           1.5            21.1    20413
Spp1+Trem2-    72.4  15.2    3.6  20.7            1.4           0.2            14.9    13061
Spp1-Trem2+    86.4  20.5    3.6  15.3            1.9           0.1            1

## 3. Figure A1 — who carries each step of the adenosine cascade

In [4]:
def panel_tag(ax, s, dx=-0.09, dy=1.06):
    ax.text(dx, dy, s, transform=ax.transAxes, fontsize=13, fontweight="bold", va="bottom", ha="left", color=INK)


SH_DP = CONTRIB.set_index(["gene", "gate"])["share_mean"]
fig = plt.figure(figsize=(15.2, 13.0))
gs = GridSpec(3, 3, figure=fig, width_ratios=[1.0, 0.95, 1.05], height_ratios=[0.78, 1.06, 1.0],
              hspace=0.46, wspace=0.42)

# ---------------- A: the gates ----------------
axA = fig.add_subplot(gs[0, 0]); panel_tag(axA, "A")
axA.set_xlim(0, 2); axA.set_ylim(0, 2); axA.set_xticks([]); axA.set_yticks([])
for sp_ in axA.spines.values(): sp_.set_visible(False)
cells = [("Spp1-Trem2+", 0, 1), ("Spp1+Trem2+", 1, 1), ("Spp1-Trem2-", 0, 0), ("Spp1+Trem2-", 1, 0)]
for k, cx, cy in cells:
    n = int(SIZES.loc[k, "n_cells"]); pct = SIZES.loc[k, "pct_of_macrophages"]
    axA.add_patch(Rectangle((cx + 0.04, cy + 0.04), 0.92, 0.92, facecolor=GCOL[k],
                            alpha=0.30 if k != "Spp1+Trem2+" else 0.55, edgecolor=GCOL[k], lw=1.6))
    axA.text(cx + 0.5, cy + 0.68, GLAB[k], ha="center", va="center", fontsize=10.5, fontweight="bold", color=INK)
    axA.text(cx + 0.5, cy + 0.44, f"{n:,} cells", ha="center", va="center", fontsize=9.5, color=INK2)
    axA.text(cx + 0.5, cy + 0.25, f"{pct:.0f}% of macrophages", ha="center", va="center", fontsize=8.5, color=MUTED)
axA.set_xlabel("Spp1 detected  →", labelpad=6, fontsize=9.5)
axA.set_ylabel("Trem2 detected  →", labelpad=6, fontsize=9.5)
axA.set_title(f"Macrophage gates ({mac.sum():,} cells)\nmarker detection, count ≥ 1", pad=10, loc="left")

# ---------------- B: the two routes, annotated with who holds each step ----------------
axB = fig.add_subplot(gs[0, 1:]); panel_tag(axB, "B", dx=-0.045)
axB.set_xlim(0, 10); axB.set_ylim(-0.3, 6.4); axB.axis("off")
axB.set_title("Where each enzymatic step is made — % of the tumor's transcripts held by Spp1⁺Trem2⁺ macrophages",
              pad=10, loc="left", fontsize=10)


def sbox(x, y, w, h, text, fc="#f3f4fb", ec="#9aa0c4", fs=9, bold=False):
    axB.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle="round,pad=0.06,rounding_size=0.12",
                                 facecolor=fc, edgecolor=ec, lw=1.2))
    axB.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK,
             fontweight="bold" if bold else "normal")


def step(x0, x1, y, gene):
    axB.add_patch(FancyArrowPatch((x0, y), (x1, y), arrowstyle="-|>", mutation_scale=13, lw=1.8,
                                  color="#2b3040", shrinkA=0, shrinkB=0))
    xm = (x0 + x1) / 2
    share = SH_DP.loc[(gene, "Spp1+Trem2+")]
    axB.text(xm, y + 0.62, ELAB[gene].replace("\n", " · "), ha="center", va="center", fontsize=8.6,
             fontweight="bold", color=INK,
             bbox=dict(boxstyle="round,pad=0.25", facecolor="#ece8fb", edgecolor="#b7aeec", lw=0.9))
    axB.text(xm, y + 0.30, ESTEP[gene], ha="center", va="center", fontsize=7.4, color=MUTED)
    bw = 0.86
    axB.add_patch(Rectangle((xm - bw/2, y - 0.62), bw, 0.15, facecolor=GRID, edgecolor="none"))
    axB.add_patch(Rectangle((xm - bw/2, y - 0.62), bw * share / 100, 0.15, facecolor=C_DP, edgecolor="none"))
    axB.text(xm, y - 0.82, f"{share:.0f}%", ha="center", va="top", fontsize=9, fontweight="bold", color=C_DP)


axB.text(0.1, 5.95, "Canonical route", fontsize=10, fontweight="bold", color=INK2)
sbox(1.25, 4.75, 2.0, 0.95, "Extracellular\nATP / ADP")
step(2.35, 3.75, 4.75, "Entpd1")
sbox(4.25, 4.75, 1.0, 0.95, "AMP", fc="#eaf5ec", ec="#8fbf9c")
step(4.85, 6.25, 4.75, "Nt5e")
sbox(7.3, 4.75, 2.0, 0.95, "Extracellular\nadenosine", fc="#fdeeec", ec="#e0a39c", bold=True)

axB.text(0.1, 2.75, "Alternative route (NAD⁺)", fontsize=10, fontweight="bold", color=INK2)
sbox(0.95, 1.55, 1.6, 0.95, "Extracellular\nNAD⁺")
step(1.85, 2.95, 1.55, "Cd38")
sbox(3.45, 1.55, 1.3, 0.95, "ADP-ribose", fc="#fdf3df", ec="#ddc07a")
step(4.15, 5.25, 1.55, "Enpp1")
sbox(5.72, 1.55, 0.95, 0.95, "AMP", fc="#eaf5ec", ec="#8fbf9c")
step(6.25, 7.35, 1.55, "Nt5e")
sbox(8.45, 1.55, 1.9, 0.95, "Extracellular\nadenosine", fc="#fdeeec", ec="#e0a39c", bold=True)
axB.text(5.0, 0.18, "The Spp1⁺Trem2⁺ gate owns the two NAD⁺ steps but not the final one — CD73 must come from a neighbor.",
         ha="center", va="center", fontsize=9, style="italic", color="#8a4b2a")

# ---------------- C: stacked contribution ----------------
axC = fig.add_subplot(gs[1, :2]); panel_tag(axC, "C", dx=-0.055)
xs = np.arange(len(ENZ)); bot = np.zeros(len(ENZ))
for k in GATES:
    v = np.array([SH_DP.loc[(g, k)] for g in ENZ])
    axC.bar(xs, v, bottom=bot, width=0.62, color=GCOL[k], edgecolor="white", lw=1.1,
            label=GLAB[k] + (" (macrophages)" if k == "Spp1+Trem2+" else ""))
    for i, (b, h) in enumerate(zip(bot, v)):
        if h >= 6:
            axC.text(xs[i] + (0.0 if k == "Spp1+Trem2+" else 0.20), b + h/2, f"{h:.0f}", ha="center", va="center", fontsize=8.5,
                     color="white" if k in (C_DP,) or k == "Spp1+Trem2+" else INK2,
                     fontweight="bold" if k == "Spp1+Trem2+" else "normal")
    bot += v
sd = np.array([CONTRIB.set_index(["gene", "gate"]).loc[(g, "Spp1+Trem2+"), "share_sd"] for g in ENZ])
dp_v = np.array([SH_DP.loc[(g, "Spp1+Trem2+")] for g in ENZ])
axC.errorbar(xs, dp_v, yerr=sd, fmt="none", ecolor="#2b1f73", elinewidth=1.4, capsize=4, capthick=1.4, zorder=5)
umi_dp = CONTRIB.set_index(["gene", "gate"]).loc[("Cd38", "Spp1+Trem2+"), "umi_share"]
axC.axhline(umi_dp, color=C_DP, ls=(0, (4, 3)), lw=1.3, zorder=4)
axC.plot([len(ENZ) - 0.6, len(ENZ) - 0.05], [umi_dp, umi_dp], color=C_DP, ls=(0, (4, 3)), lw=1.3, zorder=4)
axC.text(len(ENZ) - 0.52, umi_dp + 2.0, "expected from\ncell mass\n"
         f"({umi_dp:.0f}% of all UMI)", ha="left", va="bottom", fontsize=8, color=C_DP, style="italic")
axC.set_xticks(xs); axC.set_xticklabels([ELAB[g].replace("\n", "\n") for g in ENZ], fontsize=9.5)
axC.set_xlim(-0.55, 4.25); axC.set_ylim(0, 103); axC.set_ylabel("% of the tumor's transcripts")
axC.set_title("Contribution to each step, mouse means (n = 12 treated mice)\nwhiskers = SD across mice for the double-positive gate",
              pad=9, loc="left")
axC.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.135),
           fontsize=8.2, columnspacing=1.1, handlelength=1.3)
for i, g in enumerate(ENZ):
    axC.text(xs[i], 101, ESTEP[g], ha="center", va="bottom", fontsize=7.6, color=MUTED)

# ---------------- D: enrichment over cell mass ----------------
axD = fig.add_subplot(gs[1, 2]); panel_tag(axD, "D", dx=-0.17)
MG = GATES[:4]
cx = CONTRIB.set_index(["gene", "gate"])
for r, g in enumerate(ENZ[::-1]):
    for j, k in enumerate(MG):
        y = r + (1.5 - j) * 0.19
        row = cx.loc[(g, k)]
        axD.plot([row.enrich_lo, row.enrich_hi], [y, y], color=GCOL[k], lw=1.5, alpha=0.75, solid_capstyle="butt")
        sig = row.p_enrich < 0.05
        axD.plot(row.enrich, y, "o", ms=7, color=GCOL[k] if sig else "white",
                 markeredgecolor=GCOL[k], markeredgewidth=1.6, zorder=5)
axD.axvline(0, color=INK2, lw=1.1)
axD.text(0.06, 3.52, "no specialization", fontsize=8, color=INK2, va="center", ha="left")
axD.set_yticks(range(len(ENZ))); axD.set_yticklabels([ELAB[g].replace("\n", " ") for g in ENZ[::-1]], fontsize=9)
axD.set_xlabel("log2 (share of enzyme ÷ share of all UMI)")
axD.set_ylim(-1.35, 3.65); axD.set_xlim(-2.9, 1.85)
for r in range(len(ENZ)):
    axD.axhspan(r - 0.42, r + 0.42, color=GRID, alpha=0.35, zorder=0)
axD.set_title("Specialization, not just mass\nfilled = sign-flip p < 0.05 over mice", pad=9, loc="left")
axD.legend(handles=[Line2D([], [], marker="o", ls="", color=GCOL[k], label=GLAB[k]) for k in MG],
           frameon=False, loc="lower left", bbox_to_anchor=(0.0, 0.0), ncol=2, fontsize=8,
           columnspacing=1.0, handletextpad=0.4)

# ---------------- E: inside the double-positive gate ----------------
axE = fig.add_subplot(gs[2, :2]); panel_tag(axE, "E", dx=-0.055)
SN = {"Apoe_high_Mac": "Apoe-high", "Fn1_high_Mac": "Fn1-high", "Arg1_high_Mac": "Arg1-high",
      "Mmp12_high_Mac": "Mmp12-high", "IFN_stimulated_Mac": "IFN-stim"}
st = STATE.copy(); ys = np.arange(len(st))[::-1]
w = 0.20
ESER = [("Entpd1", "#3f8f6f"), ("Cd38", "#7067da"), ("Enpp1", "#382887"), ("Nt5e", "#d75853")]
for j, (g, c_) in enumerate(ESER):
    axE.barh(ys + (1.5 - j) * w, st[g].values, height=w, color=c_, edgecolor="white", lw=0.8,
             label=f"{ELAB[g].replace(chr(10), ' ')}")
    for y_, v_ in zip(ys + (1.5 - j) * w, st[g].values):
        axE.text(v_ + 1.0, y_, f"{v_:.0f}", va="center", ha="left", fontsize=7.6, color=INK2)
axE.set_yticks(ys); axE.set_yticklabels([f"{SN[i]}\n{int(st.loc[i,'n_cells']):,} cells" for i in st.index], fontsize=8.5)
axE.set_xlabel("% of cells in the gate detecting the enzyme"); axE.set_xlim(0, 108)
axE.set_title("Inside the Spp1⁺Trem2⁺ gate the division of labor persists\n"
              "CD39 is on nearly every cell; Apoe-high cells add the NAD⁺ steps, Arg1-high and Fn1-high cells carry CD73",
              pad=9, loc="left")
axE.legend(frameon=False, ncol=4, loc="lower right", fontsize=8.2, columnspacing=1.1, handlelength=1.3)

# ---------------- F: completeness ----------------
axF = fig.add_subplot(gs[2, 2]); panel_tag(axF, "F", dx=-0.17)
feats = [("Entpd1", "CD39⁺"), ("Nt5e", "CD73⁺"), ("canon_complete", "CD39⁺\n+CD73⁺"),
         ("both_upstream", "CD38⁺\n+CD203a⁺"), ("alt_complete", "all three\n(NAD⁺ route)")]
xf = np.arange(len(feats)); wf = 0.19
EXPECT = {"canon_complete": ("Entpd1", "Nt5e"), "both_upstream": ("Cd38", "Enpp1"),
          "alt_complete": ("Cd38", "Enpp1", "Nt5e")}
for j, k in enumerate(MG):
    v = [DET.loc[k, f] for f, _ in feats]
    axF.bar(xf + (j - 1.5) * wf, v, width=wf, color=GCOL[k], edgecolor="white", lw=0.8)
    for i, (f, _) in enumerate(feats):
        if f not in EXPECT: continue
        e = np.prod([DET.loc[k, g] / 100 for g in EXPECT[f]]) * 100
        axF.plot([xf[i] + (j - 1.5) * wf - wf * 0.46, xf[i] + (j - 1.5) * wf + wf * 0.46], [e, e],
                 color=INK, lw=1.3, zorder=6, solid_capstyle="butt")
for i, (f, _) in enumerate(feats):
    axF.text(xf[i] - 1.5 * wf, DET.loc["Spp1+Trem2+", f] + 1.6, f"{DET.loc['Spp1+Trem2+', f]:.1f}%",
             ha="center", va="bottom", fontsize=7.8, color=C_DP, fontweight="bold")
axF.set_xticks(xf); axF.set_xticklabels([l for _, l in feats], fontsize=7.6)
axF.set_ylabel("% of cells in the gate"); axF.set_ylim(0, 100)
axF.set_title("CD39 is near-universal; CD73 is the bottleneck\n"
              "bars = cells carrying the steps, dashes = expected if independent", pad=9, loc="left")

fig.suptitle("Adenosine production by Spp1 / Trem2 macrophage gates — single cell, treated arms",
             fontsize=14, fontweight="bold", x=0.09, ha="left", y=0.975)
fig.text(0.09, 0.945,
         "Gates are per-cell marker detection within macrophages. Contribution = the gate's share of the whole tumor's transcripts for that enzyme "
         "(a partition over all cells), averaged within mouse then across the 12 treated mice.",
         fontsize=9, color=INK2, ha="left")
save_figure(fig, "FigureA1_spp1_trem2_adenosine")
plt.close(fig)

saved /Users/janzules/Roselab/Spatial/CAR_T/results/rt_advantage/specific_cell_type/adenosine/FigureA1_spp1_trem2_adenosine.pdf / .png


## 4. Summary

In [5]:
cx = CONTRIB.set_index(["gene", "gate"])
print("SUMMARY — report these back")
print("=" * 78)
print(f"Gates (macrophages only, marker detection): " +
      ", ".join(f"{GLAB[k]} {int(SIZES.loc[k,'n_cells']):,} ({SIZES.loc[k,'pct_of_macrophages']:.0f}%)"
                for k in GATES[:4]))
print(f"The double-positive gate is {cx.loc[('Cd38','Spp1+Trem2+'),'umi_share']:.0f}% of the tumor's "
      f"transcriptional mass, so that is the no-specialization baseline.\n")
for g in ENZ:
    r = cx.loc[(g, "Spp1+Trem2+")]
    print(f"{ELAB[g].replace(chr(10),' '):16s} Spp1+Trem2+ holds {r.share_mean:5.1f}% "
          f"(95% CI {r.share_lo:.1f}-{r.share_hi:.1f}) | enrichment {r.enrich:+.2f} log2 "
          f"(CI {r.enrich_lo:+.2f},{r.enrich_hi:+.2f}; sign-flip p {r.p_enrich:.3f})")
print()
print("Division of labor: the gate owns the two NAD+ steps (CD38, CD203a) and is only at parity for")
print("CD39 and CD73 — those two are held in proportion to cell mass by everything in the tumor.")
print(f"Within the gate, Apoe-high cells are {STATE.loc['Apoe_high_Mac','Enpp1']:.0f}% CD203a+ but only "
      f"{STATE.loc['Apoe_high_Mac','Nt5e']:.0f}% CD73+, while Arg1-high cells are "
      f"{STATE.loc['Arg1_high_Mac','Nt5e']:.0f}% CD73+ and {STATE.loc['Arg1_high_Mac','Enpp1']:.0f}% CD203a+.")
_e = DET.loc['Spp1+Trem2+','Entpd1']/100 * DET.loc['Spp1+Trem2+','Nt5e']
print(f"CD39 is on {DET.loc['Spp1+Trem2+','Entpd1']:.0f}% of double-positive cells (the highest of any gate) but is "
      f"not enriched over cell mass; {DET.loc['Spp1+Trem2+','canon_complete']:.0f}% carry CD39 and CD73 together,")
print(f"which is exactly the {_e:.0f}% expected if the two were independent — there is no CD39/CD73 double-positive subset.")
print(f"Only {DET.loc['Spp1+Trem2+','alt_complete']:.1f}% of double-positive cells carry all three "
      f"alternative-route enzymes, and {DET.loc['Spp1+Trem2+','both_upstream']:.1f}% carry the two upstream ones:")
print("no cell makes adenosine on its own, which is the relay.")
print()
print("Sensitivity (Spp1-high = top within-sample tertile instead of bare detection):")
print("  Spp1+Trem2+ holds " + ", ".join(f"{SENS.loc['Spp1+Trem2+',g]:.0f}% of {g}" for g in ENZ) +
      " — same ranking, smaller gate.")
print()
print("Caveats: transcript level, adenosine itself is never measured; Spp1 detection in macrophages")
print("carries ambient from the Spp1-high tumor cells, which is why the tertile sensitivity is reported;")
print("the gates are descriptive, not treatment contrasts.")
print("=" * 78)

SUMMARY — report these back
Gates (macrophages only, marker detection): Spp1⁺Trem2⁺ 20,413 (42%), Spp1⁺Trem2⁻ 13,061 (27%), Spp1⁻Trem2⁺ 5,247 (11%), Spp1⁻Trem2⁻ 9,754 (20%)
The double-positive gate is 33% of the tumor's transcriptional mass, so that is the no-specialization baseline.

CD39 ENTPD1      Spp1+Trem2+ holds  36.8% (95% CI 27.7-45.9) | enrichment +0.11 log2 (CI -0.05,+0.27; sign-flip p 0.151)
CD38             Spp1+Trem2+ holds  59.2% (95% CI 52.3-66.0) | enrichment +0.88 log2 (CI +0.75,+1.01; sign-flip p 0.000)
CD203a ENPP1     Spp1+Trem2+ holds  71.0% (95% CI 64.4-77.7) | enrichment +1.15 log2 (CI +1.01,+1.28; sign-flip p 0.000)
CD73 NT5E        Spp1+Trem2+ holds  37.0% (95% CI 32.0-42.0) | enrichment +0.19 log2 (CI -0.06,+0.44; sign-flip p 0.120)

Division of labor: the gate owns the two NAD+ steps (CD38, CD203a) and is only at parity for
CD39 and CD73 — those two are held in proportion to cell mass by everything in the tumor.
Within the gate, Apoe-high cells are 27% CD203